In [ ]:
# pip install ucimlrepo

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from knn import KnnClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from itertools import zip_longest
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
breast_cancer_wisconsin_original = fetch_ucirepo(id=15) 
  
# data (as pandas dataframes) 
X = breast_cancer_wisconsin_original.data.features 
y = breast_cancer_wisconsin_original.data.targets 
  

X_std = (X - X.mean()) / X.std()
y['Class'] = np.where(y == 4, 1, 0)

In [ ]:
def cols_from_combination_bitfield(bitfield, set):
    return [a[1] for a in filter(lambda a: bitfield & (2 ** a[0]) != 0, enumerate(set))]

In [ ]:
best1 = cols_from_combination_bitfield(191, X.columns)
best2 = cols_from_combination_bitfield(255, X.columns)
best3 = cols_from_combination_bitfield(253, X.columns)

In [ ]:
pipeline = Pipeline([
    ("knn", KnnClassifier())
])

param_grid = {
    "k": range(1, 31)
}

grid_search = GridSearchCV(
    estimator=KnnClassifier(),
    param_grid=param_grid,
    cv=10,
    scoring="f1",
    n_jobs=-1
)

def train_feature_set(X, y):
    X.dropna(axis=0, inplace=True)
    y = y.iloc[list(X.index)]

    grid_search.fit(X, y)
    return pd.DataFrame(grid_search.cv_results_)

def plot_cv_results(results):
    plt.plot(
        results["param_k"],
        results["mean_test_score"],
        marker="o"
    )

    plt.xlabel("Number of neighbors (k)")
    plt.ylabel("Cross-validation accuracy")
    plt.title("KNN Hyperparameter Tuning")
    plt.show()

In [ ]:
results = train_feature_set(X, y)
plot_cv_results(results)

In [ ]:
display(best1)
results = train_feature_set(X[best1], y)
plot_cv_results(results)

In [ ]:
display(best2)
results = train_feature_set(X[best2], y)
plot_cv_results(results)

In [ ]:
display(best3)
results = train_feature_set(X[best3], y)
plot_cv_results(results)

In [ ]:
def plot_multiple_cv_results(results, titles=[]):
    for result, title in zip_longest(results, titles):
        plt.plot(
            result["param_k"],
            result["mean_test_score"],
            marker="o",
            label=title
        )

    plt.xlabel("Number of neighbors (k)")
    plt.ylabel("Cross-validation F1 score")
    plt.title(f"KNN Hyperparameter Tuning")
    if titles:
        plt.legend()
    plt.show()

plt.figure(figsize=(8, 5))

results_all = train_feature_set(X, y)
results_subset = train_feature_set(X[best1], y)
plot_multiple_cv_results([results_all, results_subset], ["All features", "Highest performing feature subset (found via brute force)"])

In [ ]:
# Best model
param_grid = {
    "k": [4]
}

grid_search = GridSearchCV(
    estimator=KnnClassifier(),
    param_grid=param_grid,
    cv=10,
    scoring="f1",
    n_jobs=-1
)

results = train_feature_set(X[best1], y)
print(f"Mean F1 Score for Best Model (10-Fold Cross-Validation): {round(results.mean_test_score.iloc[0], 3)}")
print(f"Columns Included: {best1}")
print(f"k: 4")

In [ ]:
# Best model
param_grid = {
    "k": [6]
}

grid_search = GridSearchCV(
    estimator=KnnClassifier(),
    param_grid=param_grid,
    cv=10,
    scoring="f1",
    n_jobs=-1
)

results = train_feature_set(X, y)
print(f"Mean F1 Score for Model with All Features (10-Fold Cross-Validation): {round(results.mean_test_score.iloc[0], 3)}")
print(f"Columns Included: {X.columns.tolist()}")
print(f"k: 6")